# Version 2 synthetic mortgage-lending exploratory analysis

This notebook connects the main Version 2 research layers:

1. synthetic applicants and fixed loan requests;
2. hidden conditional repayment risk and realized repayment outcomes;
3. traditional versus flexible-ML risk estimates;
4. budget-constrained portfolio allocation;
5. direct belief distortion and upstream opportunity mechanisms; and
6. the corrected five-replication systemic checkpoint.

The purpose is descriptive and diagnostic. The notebook does not train models, regenerate data, rerun optimization, or modify source artifacts.

### Interpretation boundaries

- Groups `A` and `B` are researcher-defined synthetic groups, not empirical racial groups.
- `rho` is the probability of making the next scheduled payment conditional on having made all previous payments.
- The oracle uses hidden simulation truth and is an evaluator benchmark, not a feasible lender.
- Direct and systemic mechanisms are present only because the simulation explicitly encodes them.
- The repeated-seed section contains five corrected replications. It is an interim checkpoint, not the planned final 50-replication analysis.

## 1. Setup and source inventory

The loader finds the repository root whether the notebook is opened from the root directory or from `notebooks/`. All inputs are read-only CSV artifacts already produced by the project pipeline.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

NAVY = "#17365D"
TEAL = "#148F88"
ORANGE = "#D97706"
GRAY = "#6B7280"
LIGHT = "#E5E7EB"

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "axes.edgecolor": "#9CA3AF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "font.size": 10,
})

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "results" / "tables").exists():
            return candidate
    raise FileNotFoundError("Could not locate the econ4994 project root.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "synthetic" / "economic_lending" / "v2_baseline"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"

SOURCES = {
    "applicants": DATA_DIR / "applicants.csv",
    "loans": DATA_DIR / "loan_options.csv",
    "truth": DATA_DIR / "simulation_truth.csv",
    "outcomes": DATA_DIR / "loan_outcomes.csv",
    "risk_additive": TABLE_DIR / "v2_risk_model_comparison.csv",
    "risk_nonlinear": TABLE_DIR / "v2_nonlinear_risk_model_comparison.csv",
    "portfolio": TABLE_DIR / "v2_portfolio_summary.csv",
    "regret": TABLE_DIR / "v2_portfolio_regret.csv",
    "direct": TABLE_DIR / "v2_direct_group_effects.csv",
    "systemic_access": TABLE_DIR / "v2_systemic_opportunity_summary.csv",
    "systemic_switchers": TABLE_DIR / "v2_systemic_switcher_effects.csv",
    "systemic_checkpoint": TABLE_DIR / "v2_systemic_mc_corrected_checkpoint_summary.csv",
}

missing = [str(path) for path in SOURCES.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required source files:\n" + "\n".join(missing))

frames = {name: pd.read_csv(path) for name, path in SOURCES.items()}
inventory = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "memory_mb": frame.memory_usage(deep=True).sum() / 1_000_000,
        "relative_path": str(SOURCES[name].relative_to(PROJECT_ROOT)),
    }
    for name, frame in frames.items()
])
inventory

## 2. Integrity checks and analysis table

The four borrower-level files should have one row per applicant and join one-to-one. The checks also verify the repayment identity

**P(complete all payments) = rho^T**

and confirm that completed loans have no recorded default period.

In [ ]:
applicants = frames["applicants"].copy()
loans = frames["loans"].copy()
truth = frames["truth"].copy()
outcomes = frames["outcomes"].copy()

checks = {
    "10,000 applicants": len(applicants) == 10_000,
    "applicant IDs unique": applicants["applicant_id"].is_unique,
    "loan IDs unique": loans["applicant_id"].is_unique,
    "truth IDs unique": truth["applicant_id"].is_unique,
    "outcome IDs unique": outcomes["applicant_id"].is_unique,
    "all applicant key sets match": (
        set(applicants.applicant_id) == set(loans.applicant_id)
        == set(truth.applicant_id) == set(outcomes.applicant_id)
    ),
    "rho lies in [0, 1]": truth["repayment_probability_per_period_true"].between(0, 1).all(),
    "full-repayment probability lies in [0, 1]": truth["full_repayment_probability_true"].between(0, 1).all(),
    "loan principal is positive": loans["requested_principal"].gt(0).all(),
    "completed loans have no default period": outcomes.loc[outcomes.completed_all_payments, "default_period"].isna().all(),
}

rho = truth["repayment_probability_per_period_true"].to_numpy()
term = loans["term_periods"].to_numpy()
checks["full repayment equals rho ** term"] = np.allclose(
    truth["full_repayment_probability_true"].to_numpy(), rho ** term, rtol=1e-12, atol=1e-12
)

check_table = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
display(check_table)
assert check_table["passed"].all(), "One or more source-integrity checks failed."

master = (
    applicants
    .merge(loans, on="applicant_id", validate="one_to_one")
    .merge(truth, on="applicant_id", validate="one_to_one")
    .merge(outcomes, on="applicant_id", validate="one_to_one")
)
master["periods_paid"] = np.where(
    master["completed_all_payments"],
    master["term_periods"],
    master["default_period"] - 1,
)
master["realized_positive_profit"] = master["realized_profit"] > 0

print(f"Merged analysis table: {master.shape[0]:,} applicants × {master.shape[1]:,} columns")
master.head()

## 3. Population composition and financial distributions

The cohort split occurs at the applicant level, which prevents payment rows from the same borrower from crossing training, validation, and evaluation samples.

In [ ]:
cohort_group = pd.crosstab(applicants["cohort"], applicants["group"], margins=True)
display(cohort_group)

summary_variables = [
    "annual_income", "credit_score", "employment_years", "liquid_assets",
    "existing_monthly_debt", "property_value", "requested_principal",
    "first_period_dti", "requested_ltv",
]
population_summary = master[summary_variables].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T
population_summary

In [ ]:
plot_specs = [
    ("annual_income", "Annual income ($)", (0.01, 0.99)),
    ("credit_score", "Credit score", (0.01, 0.99)),
    ("liquid_assets", "Liquid assets ($)", (0.01, 0.99)),
    ("requested_principal", "Requested principal ($)", (0.01, 0.99)),
    ("first_period_dti", "First-period DTI", (0.01, 0.99)),
    ("requested_ltv", "Requested LTV", (0.01, 0.99)),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (column, label, quantiles) in zip(axes.ravel(), plot_specs):
    low, high = master[column].quantile(list(quantiles))
    clipped = master[column].clip(low, high)
    ax.hist(clipped, bins=40, color=TEAL, alpha=0.85, edgecolor="white")
    ax.set_title(label)
    ax.set_ylabel("Applicants")
    if column in {"first_period_dti", "requested_ltv"}:
        ax.xaxis.set_major_formatter(PercentFormatter(1.0))
fig.suptitle("Baseline borrower and loan distributions (1st–99th percentile display range)", fontsize=14, color=NAVY)
fig.tight_layout()
plt.show()

### Baseline group-balance diagnostic

For each continuous variable, the standardized mean difference is

**Standardized mean difference = (mean_B - mean_A) / pooled SD**

This is a descriptive balance check. It is not a test of real-world group equality.

In [ ]:
balance_variables = [
    "age_years", "annual_income", "credit_score", "employment_years",
    "liquid_assets", "existing_monthly_debt", "property_value",
    "requested_principal", "first_period_dti", "requested_ltv",
    "repayment_probability_per_period_true", "full_repayment_probability_true",
]

def standardized_mean_difference(frame: pd.DataFrame, column: str) -> float:
    grouped = frame.groupby("group", observed=True)[column]
    means = grouped.mean()
    variances = grouped.var(ddof=1)
    pooled_sd = np.sqrt((variances.loc["A"] + variances.loc["B"]) / 2)
    return float((means.loc["B"] - means.loc["A"]) / pooled_sd)

balance = pd.DataFrame({
    "variable": balance_variables,
    "group_a_mean": [master.loc[master.group.eq("A"), c].mean() for c in balance_variables],
    "group_b_mean": [master.loc[master.group.eq("B"), c].mean() for c in balance_variables],
    "smd_b_minus_a": [standardized_mean_difference(master, c) for c in balance_variables],
})
balance["absolute_smd"] = balance["smd_b_minus_a"].abs()
display(balance.sort_values("absolute_smd", ascending=False))

ordered = balance.sort_values("smd_b_minus_a")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(ordered["variable"], ordered["smd_b_minus_a"], color=np.where(ordered.smd_b_minus_a >= 0, TEAL, ORANGE))
ax.axvline(0, color="black", linewidth=1)
ax.axvline(-0.10, color=GRAY, linestyle="--", linewidth=1)
ax.axvline(0.10, color=GRAY, linestyle="--", linewidth=1)
ax.set_xlabel("Standardized mean difference (Group B − Group A)")
ax.set_title("Baseline group balance across borrower, loan, and risk variables", color=NAVY)
fig.tight_layout()
plt.show()

print(f"Largest absolute baseline SMD: {balance.absolute_smd.max():.3f}")

## 4. Hidden repayment risk and realized outcomes

The true per-period probability is close to one for most applicants, but a 120-period horizon compounds small differences. Consequently, whole-loan completion probabilities are much more dispersed than per-period probabilities.

In [ ]:
risk_outcome_summary = pd.DataFrame({
    "metric": [
        "Mean conditional repayment probability",
        "Mean full-repayment probability",
        "Observed full-completion rate",
        "Mean periods paid",
        "Mean realized profit",
        "Share with positive realized profit",
    ],
    "value": [
        master.repayment_probability_per_period_true.mean(),
        master.full_repayment_probability_true.mean(),
        master.completed_all_payments.mean(),
        master.periods_paid.mean(),
        master.realized_profit.mean(),
        master.realized_positive_profit.mean(),
    ],
})
display(risk_outcome_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(master.repayment_probability_per_period_true, bins=45, color=TEAL, alpha=0.85)
axes[0].set_title("Conditional repayment probability")
axes[0].set_xlabel("True per-period rho")
axes[0].set_ylabel("Applicants")
axes[1].hist(master.full_repayment_probability_true, bins=45, color=ORANGE, alpha=0.85)
axes[1].set_title("Whole-contract completion probability")
axes[1].set_xlabel("True rho raised to 120")
fig.tight_layout()
plt.show()

In [ ]:
calibration = master.copy()
calibration["risk_bin"] = pd.qcut(
    calibration["full_repayment_probability_true"], q=10, duplicates="drop"
)
completion_calibration = (
    calibration.groupby("risk_bin", observed=True)
    .agg(
        n=("applicant_id", "size"),
        mean_true_full_repayment=("full_repayment_probability_true", "mean"),
        observed_completion_rate=("completed_all_payments", "mean"),
    )
    .reset_index(drop=True)
)
display(completion_calibration)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(
    completion_calibration.mean_true_full_repayment,
    completion_calibration.observed_completion_rate,
    marker="o", color=TEAL, linewidth=2,
)
ax.plot([0, 1], [0, 1], linestyle="--", color=GRAY, label="45-degree reference")
ax.set(
    xlabel="Mean true full-repayment probability",
    ylabel="Observed completion rate",
    title="Realized completion tracks synthetic truth by decile",
    xlim=(0, 1), ylim=(0, 1),
)
ax.legend()
fig.tight_layout()
plt.show()

## 5. Risk-model recovery

Two evaluations are intentionally separate:

- probability recovery compares lender estimates with hidden `rho_true`;
- realized-label metrics compare probabilities with stochastic next-payment outcomes.

Even the oracle cannot perfectly predict Bernoulli outcomes.

In [ ]:
additive_risk = frames["risk_additive"].query("cohort == 'evaluation'").copy()
additive_risk["world"] = "Additive"
additive_risk["model_label"] = additive_risk["model"].replace({
    "traditional_logit_v1": "Traditional",
    "ml_histgb_v1": "ML",
    "oracle": "Oracle",
})

nonlinear_risk = frames["risk_nonlinear"].query("cohort == 'evaluation'").copy()
nonlinear_risk["world"] = "Nonlinear"
nonlinear_risk["model_label"] = nonlinear_risk["model"].replace({
    "traditional": "Traditional", "ml": "ML", "oracle": "Oracle"
})

risk_comparison = pd.concat([
    additive_risk[["world", "model_label", "mae", "rmse", "brier_score", "log_loss", "roc_auc"]],
    nonlinear_risk[["world", "model_label", "mae", "rmse", "brier_score", "log_loss", "roc_auc"]],
], ignore_index=True)
display(risk_comparison)

model_only = risk_comparison[risk_comparison.model_label.ne("Oracle")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric, title in [
    (axes[0], "mae", "Error versus hidden conditional risk"),
    (axes[1], "brier_score", "Error versus realized next-payment labels"),
]:
    pivot = model_only.pivot(index="world", columns="model_label", values=metric).loc[["Additive", "Nonlinear"]]
    pivot[["Traditional", "ML"]].plot.bar(ax=ax, color=[NAVY, ORANGE], width=0.72)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("MAE" if metric == "mae" else "Brier score")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="")
fig.tight_layout()
plt.show()

## 6. Prediction error and portfolio value

The portfolio optimizer maximizes perceived expected profit subject to the lending budget. Regret evaluates the selected portfolio using hidden truth:

**Regret = oracle true expected profit - policy true expected profit.**

Lower regret is better. The oracle has zero regret by construction.

In [ ]:
portfolio = frames["portfolio"].copy()
regret = frames["regret"].copy()

portfolio_view = portfolio[[
    "world_id", "budget_id", "policy", "applicants_funded", "funding_rate",
    "budget_utilization", "true_expected_portfolio_profit",
    "truly_nonpositive_funded_fraction",
]].copy()
display(portfolio_view)

regret_models = regret[regret.policy.ne("oracle")].copy()
budget_order = ["nonbinding_100pct", "moderate_40pct", "tight_20pct"]
world_labels = {
    "additive_logistic_baseline": "Additive",
    "nonlinear_v1": "Nonlinear",
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, (world_id, frame) in zip(axes, regret_models.groupby("world_id", sort=False)):
    pivot = frame.pivot(index="budget_id", columns="policy", values="regret_percent_oracle").reindex(budget_order)
    pivot[["traditional", "ml"]].plot.bar(ax=ax, color=[NAVY, ORANGE], width=0.72)
    ax.set_title(f"{world_labels[world_id]} risk world")
    ax.set_xlabel("")
    ax.set_ylabel("Regret (% of oracle profit)")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(["Traditional", "ML"], title="")
fig.suptitle("Economic regret depends on the risk world and capital constraint", color=NAVY, fontsize=14)
fig.tight_layout()
plt.show()

best_by_cell = (
    regret_models.sort_values("regret_dollars")
    .groupby(["world_id", "budget_id"], as_index=False)
    .first()[["world_id", "budget_id", "policy", "regret_dollars", "regret_percent_oracle"]]
)
best_by_cell = best_by_cell.rename(columns={"policy": "lower_regret_policy"})
best_by_cell

## 7. Direct belief distortion

The direct experiment reduces Group B’s perceived repayment probability on the log-odds scale while leaving true borrower risk unchanged:

**logit(rho_used) = logit(rho_base) - delta x I(Group B).**

The plots show the change in the Group B-minus-Group A funding gap relative to the same policy at `delta = 0`.

In [ ]:
direct = frames["direct"].copy()
direct_models = direct[direct.family.isin(["traditional", "ml"])].copy()

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
for ax, ((world, budget), frame) in zip(
    axes.ravel(), direct_models.groupby(["world_id", "budget_id"], sort=False)
):
    for family, color, label in [
        ("traditional", NAVY, "Traditional"),
        ("ml", ORANGE, "ML"),
    ]:
        series = frame[frame.family.eq(family)].sort_values("delta")
        ax.plot(series.delta, 100 * series.direct_effect_on_funding_gap, marker="o", color=color, label=label)
    ax.axhline(0, color=GRAY, linewidth=1)
    ax.set_title(f"{world_labels[world]}\n{budget}")
    ax.set_xlabel("Direct penalty delta (log-odds)")
    ax.set_ylabel("Change in B−A funding gap (pp)")
axes[0, 0].legend()
fig.suptitle("Direct belief distortion shifts funding away from Group B", color=NAVY, fontsize=14)
fig.tight_layout()
plt.show()

direct_020 = direct_models[direct_models.delta.eq(0.20)][[
    "world_id", "family", "budget_id",
    "group_a_funding_rate_change", "group_b_funding_rate_change",
    "direct_effect_on_funding_gap",
]].copy()
for column in ["group_a_funding_rate_change", "group_b_funding_rate_change", "direct_effect_on_funding_gap"]:
    direct_020[column] *= 100
direct_020.rename(columns={
    "group_a_funding_rate_change": "group_a_change_pp",
    "group_b_funding_rate_change": "group_b_change_pp",
    "direct_effect_on_funding_gap": "gap_change_pp",
})

## 8. Upstream opportunity mechanism

The systemic experiment acts earlier in the causal chain. It lowers Group B’s probability of receiving a favorable opportunity state, which then changes financial characteristics, true repayment risk, expected profit, and potentially funding. There is no direct Group B term in the lender’s final repayment decision in this mechanism.

In [ ]:
systemic_access = frames["systemic_access"].copy()
switchers = frames["systemic_switchers"].copy()

fig, ax = plt.subplots(figsize=(7.5, 4.8))
for group, color in [("A", NAVY), ("B", ORANGE)]:
    frame = systemic_access[systemic_access.group.eq(group)].sort_values("systemic_strength")
    ax.plot(frame.systemic_strength, 100 * frame.opportunity_access_rate, marker="o", linewidth=2, color=color, label=f"Group {group}")
ax.set(
    xlabel="Systemic strength s (Group B opportunity log-odds penalty)",
    ylabel="Opportunity access rate (%)",
    title="Upstream treatment reduces Group B opportunity access",
)
ax.legend()
fig.tight_layout()
plt.show()

switcher_040 = switchers[
    switchers.systemic_strength.eq(0.40) & switchers.population.eq("group_b_switchers")
][[
    "risk_world", "n", "share_of_group_b",
    "mean_change_employment_years", "mean_change_annual_income",
    "mean_change_liquid_assets", "mean_change_property_value",
    "mean_change_requested_loan_amount", "mean_change_first_period_dti",
    "mean_change_requested_ltv", "mean_change_full_repayment_probability_true",
    "mean_change_expected_profit_true",
]].copy()
switcher_display = switcher_040.set_index("risk_world").T
switcher_display.index.name = "metric"
display(switcher_display)

## 9. Corrected repeated-seed systemic checkpoint

This section summarizes five validated corrected replications. Error bars show Monte Carlo standard errors, not confidence intervals about a real population. With only five replications, signs can remain unstable and model rankings should not be generalized.

In [ ]:
checkpoint = frames["systemic_checkpoint"].copy()
assert checkpoint.n_corrected.eq(5).all()
assert checkpoint.evidence_status.eq("INTERIM CORRECTED 5-REPLICATION SUMMARY").all()

gap_checkpoint = checkpoint[
    checkpoint.estimand.eq("delta_funding_gap")
    & checkpoint.systemic_strength.eq(0.40)
    & checkpoint.budget_id.eq("moderate_40pct")
].copy()
gap_checkpoint["mean_pp"] = 100 * gap_checkpoint["mean"]
gap_checkpoint["mcse_pp"] = 100 * gap_checkpoint["mcse"]
gap_checkpoint["minimum_pp"] = 100 * gap_checkpoint["minimum"]
gap_checkpoint["maximum_pp"] = 100 * gap_checkpoint["maximum"]
gap_checkpoint["label"] = gap_checkpoint["risk_world"].map(world_labels) + " | " + gap_checkpoint["family"].replace({
    "traditional": "Traditional", "ml": "ML", "true_risk_reference": "Oracle"
})
display(gap_checkpoint[["label", "n_corrected", "mean_pp", "mcse_pp", "minimum_pp", "maximum_pp"]])

plot_frame = gap_checkpoint.sort_values(["risk_world", "family"])
fig, ax = plt.subplots(figsize=(9, 5))
colors = plot_frame.family.map({"traditional": NAVY, "ml": ORANGE, "true_risk_reference": TEAL})
ax.barh(plot_frame.label, plot_frame.mean_pp, xerr=plot_frame.mcse_pp, color=colors, alpha=0.88, capsize=4)
ax.axvline(0, color="black", linewidth=1)
ax.set(
    xlabel="Mean change in B−A funding gap (percentage points, ±1 MCSE)",
    title="Interim corrected checkpoint: s = 0.40, 40% budget",
)
fig.tight_layout()
plt.show()

## 10. Overall findings from the current synthetic snapshot

The following cell derives a concise set of findings directly from the loaded files, so the numbers update when the notebook is rerun against compatible artifacts.

In [ ]:
max_smd = balance.absolute_smd.max()
expected_completion = master.full_repayment_probability_true.mean()
observed_completion = master.completed_all_payments.mean()

additive_models = additive_risk[additive_risk.model_label.isin(["Traditional", "ML"])].set_index("model_label")
nonlinear_models = nonlinear_risk[nonlinear_risk.model_label.isin(["Traditional", "ML"])].set_index("model_label")

lower_regret_counts = best_by_cell.lower_regret_policy.value_counts().to_dict()

direct_example = direct_models[
    direct_models.world_id.eq("additive_logistic_baseline")
    & direct_models.family.eq("traditional")
    & direct_models.budget_id.eq("moderate_40pct")
    & direct_models.delta.eq(0.20)
].iloc[0]

access_b = systemic_access[systemic_access.group.eq("B")].set_index("systemic_strength")
switcher_example = switcher_040[switcher_040.risk_world.eq("additive_logistic_baseline")].iloc[0]

findings = [
    f"Baseline balance is strong by construction: the largest absolute Group B-minus-A standardized mean difference is {max_smd:.3f}.",
    f"The mean true whole-loan completion probability is {expected_completion:.1%}; the realized completion rate is {observed_completion:.1%}. The small difference is compatible with Bernoulli sampling variation.",
    f"In the additive world, traditional-logit risk MAE is {additive_models.loc['Traditional', 'mae']:.6f}, versus {additive_models.loc['ML', 'mae']:.6f} for ML.",
    f"In the nonlinear world, traditional risk MAE is {nonlinear_models.loc['Traditional', 'mae']:.6f}, versus {nonlinear_models.loc['ML', 'mae']:.6f} for ML. Flexibility does not guarantee better recovery under the current sample and tuning design.",
    f"Traditional has lower portfolio regret in {lower_regret_counts.get('traditional', 0)} of the 6 world-budget cells; ML has lower regret in {lower_regret_counts.get('ml', 0)}. The exception shows that model rankings can depend on the economic decision environment.",
    f"At direct penalty delta=0.20 in the additive traditional 40% budget cell, Group A funding changes by {100*direct_example.group_a_funding_rate_change:+.2f} pp, Group B by {100*direct_example.group_b_funding_rate_change:+.2f} pp, and the B-minus-A gap changes by {100*direct_example.direct_effect_on_funding_gap:+.2f} pp.",
    f"At systemic strength 0.40, Group B opportunity access falls from {access_b.loc[0.0, 'opportunity_access_rate']:.1%} to {access_b.loc[0.4, 'opportunity_access_rate']:.1%}. Among {int(switcher_example['n'])} switchers, mean income changes by ${switcher_example.mean_change_annual_income:,.0f}, liquid assets by ${switcher_example.mean_change_liquid_assets:,.0f}, and employment by {switcher_example.mean_change_employment_years:.2f} years.",
    "The corrected repeated-seed checkpoint has five replications. It is useful for identifying instability, but it is not sufficient for final uncertainty or model-ranking claims.",
]

print("\n".join(f"{index}. {finding}" for index, finding in enumerate(findings, start=1)))

## 11. Research interpretation

The current results support three distinctions:

1. **Risk prediction and economic value are related but not identical.** A model with lower probability error usually has an advantage, but the portfolio objective depends on ranking, loan size, expected profit, and the budget constraint.
2. **Direct and upstream mechanisms answer different questions.** Direct belief distortion changes the lender’s treatment of otherwise comparable applicants. The systemic mechanism changes the economic conditions applicants bring into underwriting.
3. **Single-population findings are not uncertainty estimates.** The corrected five-replication checkpoint begins to show seed-to-seed variation, but the predeclared 50-replication study remains incomplete.

These are properties of the researcher-defined synthetic environment. They do not establish discrimination in real mortgage markets, and adjusted or unadjusted differences should not automatically be interpreted as causal estimates.